# 4.7 一次訓練多個位置怎麼對齊答案？


模型已能替整段輸入的每個位置產生分數，但它究竟該跟哪個答案比？回到 [1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html) 的下一字對齊，原序列 `[1,2,3,4]` 應建立輸入 `[1,2,3]`、答案 `[2,3,4]`。第0位置看到1就猜2，第1位置已能讀到1、2而猜3，第2位置讀到1、2、3而猜4。候選軸上的ID與原文字位置不要混為同一個編號。

訓練時整份正確序列已經給定，所以可以一次輸入三個位置，在 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的因果許可下平行計算三個預測；每個位置都只能讀自己及更早的位置：第0位置只能讀輸入1，第1位置只能讀1、2，第2位置才可讀1、2、3。這樣才不會因為整段同時放入而偷看後面的答案。生成新內容時則還不知道下一字，所以仍需按 [1.14](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html) 逐步延長。資料知道答案和模型有權看到答案，是兩件事。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.data import shifted
from tiny_perceptron.model import TinyLM, ModelConfig, masked_loss

torch.manual_seed(42)
x, y = shifted([1, 2, 3, 4])
model = TinyLM(ModelConfig(vocab_size=5, width=8))
logits = model(x[None])["logits"]
loss = masked_loss(logits, y[None])
print("問題答案", list(zip(x.tolist(), y.tolist())))
print("分數形狀", logits.shape, "平均代價", loss.item())
loss.backward()

`shifted` 是專案把一次右移對齊封裝成的工具，回傳長整數tensor，因為ID是索引而非小數特徵。`x[None]` 新加筆數軸，把 `[3]` 變 `[1,3]`，模型輸出 `[1,3,5]`。`y[None]` 對應一筆三個答案。第一行應印 `(1,2),(2,3),(3,4)`，第二行確認候選數5；代價是三個位置的交叉熵平均，含義見 [1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)，未訓練時通常為正。

`masked_loss` 名字中的masked表示這工具還能忽略某些答案，本例沒有忽略項，所以三題都計入。它用相同位置的logits與y比，不會再幫你shift。每個參數都是可調的數字；[梯度](https://birdhackor.github.io/tiny-perceptron-vlm/1.9.html)記錄它稍微改動時，代價會往哪個方向變、變得多快。`backward()`只計算並保存這些影響量，不會改動參數。若要讓模型學習，還要另做[1.12的參數更新](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)；基本做法是 `新參數=舊參數−學習率×梯度`，由學習率控制這次調多少。本例到計算梯度為止，目的是核對從答案代價到可調參數的計算能接通。

若資料工具已shift一次，模型內再把答案右移，就會把「看到1猜2」改成「看到1猜3」，甚至兩端長度不同。注意力mask再正確也補救不了錯誤的訓練目標。你應先逐格列出問題答案再看loss，不能期待一個合理數字自動證明對齊無誤。

這份輸入還只用了四種小ID，並未接自然語料；我們刻意把編碼、對齊與模型接口拆開，讓第一次完整實驗不用下載資料。以後從文件切窗口時，也要保持每筆原序列先取成x、y再合批的約定。

把ID換成真的文字，這個對齊規則仍然一樣。[T.4的實跑](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)用12篇`color=red;shape=circle;side=left.`這類短文，讓每個位置猜下一個文字單位，再用9篇訓練、1篇驗證、2篇最後檢查。這份模型另做了防偷看核對：[公開原始報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/text_foundation.json)中的 `results.causal_max_difference` 記錄其結果。檢查輸入是 `[1,10,11,12,13]`，只把最後ID改成14，逐格比較前四個位置的所有候選分數，再取差的絕對值中最大的數；本次為0.0，表示這四個位置的分數沒有受到後面那格改動影響。這把「答案只移一次」與「前面不能偷看後面」接回同一個完整流程；單看loss下降，不能代替這兩項核對。

練習先將原序列延長成 `[1,2,3,4,0]`，手寫四對 `(1,2),(2,3),(3,4),(4,0)`，再執行核對輸出 `[1,4,5]`。接著只故意把 `y` 改為 `x.clone()`，長度仍可匹配、程式仍能跑，卻會學複製目前字。逐對讀輸出，才是辨認這種語意錯誤的依據。
